In [48]:
import pandas as pd
from sklearn.model_selection import train_test_split
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report


In [49]:
random_state = 42

In [50]:
df = pd.read_csv('../Milestone 3 EDA/ppp_cleaned.csv')
df

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
0,0.0,48.0,2.0,24,769358.78,769358.78,11.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,12409.01,0,1588291200,1605830400,1608249600
1,0.0,48.0,2.0,24,736927.79,736927.79,11.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,10094.90,0,1588291200,1628726400,1632787200
2,0.0,48.0,2.0,24,691355.00,691355.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9218.07,0,1588291200,1612915200,1615939200
3,0.0,48.0,2.0,24,499871.00,499871.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,23803.38,0,1588291200,1631232000,1634342400
4,0.0,48.0,2.0,24,367437.00,367437.00,37.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,14697.48,0,1588291200,1617840000,1629158400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968527,0.0,56.0,2.0,24,150000.00,150000.00,54.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10000.00,0,1585872000,1607472000,1610496000
968528,0.0,56.0,2.0,24,150000.00,150000.00,31.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,3452.38,0,1586822400,1604361600,1607385600
968529,1.0,56.0,2.0,60,150000.00,150000.00,54.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,29999.40,0,1613088000,1629158400,1631664000
968530,0.0,56.0,2.0,60,150000.00,150000.00,18.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21428.57,0,1586908800,1645574400,1646697600


In [51]:
fraud = df[df['Fraud'] == 1]
non_fraud = df[df['Fraud'] == 0]
len(fraud), len(non_fraud)

(95, 968437)

In [52]:
# Undersample majority class to a 1:10 ratio
non_fraud_downsampled = non_fraud.sample(n=len(fraud) * 10, random_state=random_state)
non_fraud_downsampled

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
495859,1.0,25.0,2.0,60,175700.0,175700.0,52.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4748.57,0,1617062400,1638835200,1641427200
751210,0.0,41.0,2.0,24,379200.0,379200.0,40.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9248.78,0,1586390400,1611619200,1613433600
302413,0.0,14.0,2.0,24,238300.0,238300.0,14.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,8510.71,0,1586217600,1605571200,1615334400
199402,0.0,8.0,2.0,24,742528.0,742528.0,37.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,19540.21,0,1586649600,1640044800,1642809600
52948,1.0,5.0,2.0,60,882672.0,882672.0,37.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,18389.00,0,1615593600,1640131200,1642809600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294662,0.0,13.0,2.0,24,298700.0,298700.0,13.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21335.71,0,1586908800,1630972800,1635379200
204016,0.0,9.0,2.0,24,375100.0,375100.0,42.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10717.14,0,1588032000,1623888000,1626912000
387773,0.0,20.0,2.0,24,603200.0,603200.0,20.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,11827.45,0,1586044800,1627948800,1631577600
378063,0.0,19.0,2.0,24,515000.0,515000.0,37.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,10510.20,0,1586995200,1614902400,1618876800


In [53]:
# Combine and shuffle
df_reduced = pd.concat([fraud, non_fraud_downsampled]).sample(frac=1).reset_index(drop=True)
df_reduced

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
0,1.0,41.0,2.0,60,398016.00,398016.00,36.0,1.0,0.0,0.0,...,0.0,0.000,0.0,0.0,0.0,8291.94,0,1613001600,1637884800,1645142400
1,0.0,24.0,2.0,24,337600.00,337600.00,42.0,1.0,1.0,1.0,...,0.0,0.000,0.0,0.0,0.0,19858.82,0,1586908800,1604361600,1608336000
2,0.0,53.0,2.0,24,191700.00,191700.00,39.0,1.0,0.0,0.0,...,0.0,0.000,0.0,0.0,0.0,9585.00,0,1588550400,1605139200,1608336000
3,1.0,37.0,1.0,60,330658.00,330658.00,40.0,1.0,0.0,1.0,...,0.0,0.000,0.0,0.0,0.0,9447.23,0,1617321600,-2208988800,1617321600
4,0.0,47.0,1.0,40,176332.50,176332.50,5.0,1.0,0.0,1.0,...,0.0,0.000,0.0,0.0,0.0,10372.50,1,1593475200,-2208988800,1593475200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1040,0.0,34.0,2.0,24,301297.00,301297.00,40.0,1.0,0.0,0.0,...,0.0,0.125,0.0,0.0,0.0,15064.87,0,1586476800,1604361600,1607731200
1041,0.0,6.0,2.0,24,174605.00,174605.00,6.0,1.0,0.0,1.0,...,0.0,0.000,0.0,0.0,0.0,15873.18,0,1587945600,1615420800,1617926400
1042,1.0,20.0,1.0,60,209351.53,209351.53,9.0,1.0,1.0,1.0,...,0.0,0.000,0.0,0.0,0.0,16103.58,0,1617235200,-2208988800,1617235200
1043,0.0,47.0,2.0,24,558700.00,558700.00,46.0,1.0,0.0,1.0,...,0.0,0.000,0.0,0.0,0.0,7906.13,0,1586304000,1607299200,1612224000


In [54]:
X = df_reduced.drop(columns='Fraud')
y = df_reduced['Fraud']

In [55]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

In [56]:
cv = StratifiedKFold(n_splits=5)

### Recall as metric

In [57]:
# Parameter tuning
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_recall_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_recall_df)

[I 2026-04-14 15:30:01,117] A new study created in memory with name: no-name-b28eea86-d191-4b47-b084-7396944ed456
[I 2026-04-14 15:30:01,205] Trial 0 finished with value: 0.9341666666666667 and parameters: {'n_estimators': 80, 'max_depth': 2, 'min_samples_split': 6}. Best is trial 0 with value: 0.9341666666666667.
[I 2026-04-14 15:30:01,392] Trial 1 finished with value: 0.8683333333333334 and parameters: {'n_estimators': 187, 'max_depth': 4, 'min_samples_split': 3}. Best is trial 0 with value: 0.9341666666666667.
[I 2026-04-14 15:30:01,550] Trial 2 finished with value: 0.9341666666666667 and parameters: {'n_estimators': 182, 'max_depth': 2, 'min_samples_split': 3}. Best is trial 0 with value: 0.9341666666666667.
[I 2026-04-14 15:30:01,612] Trial 3 finished with value: 0.8033333333333335 and parameters: {'n_estimators': 51, 'max_depth': 21, 'min_samples_split': 7}. Best is trial 0 with value: 0.9341666666666667.
[I 2026-04-14 15:30:01,814] Trial 4 finished with value: 0.8433333333333334

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
0,0,0.934167,2026-04-14 15:30:01.119071,2026-04-14 15:30:01.205715,0 days 00:00:00.086644,2,6,80,COMPLETE
2,2,0.934167,2026-04-14 15:30:01.392452,2026-04-14 15:30:01.550170,0 days 00:00:00.157718,2,3,182,COMPLETE
8,8,0.934167,2026-04-14 15:30:01.970625,2026-04-14 15:30:02.049806,0 days 00:00:00.079181,2,4,86,COMPLETE
11,11,0.934167,2026-04-14 15:30:02.270082,2026-04-14 15:30:02.394695,0 days 00:00:00.124613,2,5,138,COMPLETE
31,31,0.934167,2026-04-14 15:30:04.584067,2026-04-14 15:30:04.724643,0 days 00:00:00.140576,2,5,142,COMPLETE


In [58]:
# Train using optimal parameters
model_1 = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_recall_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_recall_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_recall_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model_1.fit(X_train, y_train)

,n_estimators,80
,criterion,'gini'
,max_depth,2
,min_samples_split,6
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [59]:
y_pred = model_1.predict(X_test)

In [60]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.99      0.91      0.95       190
           1       0.50      0.89      0.64        19

    accuracy                           0.91       209
   macro avg       0.74      0.90      0.79       209
weighted avg       0.94      0.91      0.92       209



In [61]:
recall_model_report_df = classification_report(y_test, y_pred, output_dict=True)

## F1 as metric

In [62]:
# Parameter tuning
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='f1',
        n_jobs=-1,
    )
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

top_5_f1_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_f1_df)

[I 2026-04-14 15:30:12,470] A new study created in memory with name: no-name-c80a011c-051b-4721-aadd-309af9a2638a
[I 2026-04-14 15:30:12,588] Trial 0 finished with value: 0.7994660734149054 and parameters: {'n_estimators': 110, 'max_depth': 22, 'min_samples_split': 4}. Best is trial 0 with value: 0.7994660734149054.
[I 2026-04-14 15:30:12,620] Trial 1 finished with value: 0.8076267281105991 and parameters: {'n_estimators': 19, 'max_depth': 11, 'min_samples_split': 9}. Best is trial 1 with value: 0.8076267281105991.
[I 2026-04-14 15:30:12,759] Trial 2 finished with value: 0.7244027123218217 and parameters: {'n_estimators': 157, 'max_depth': 3, 'min_samples_split': 6}. Best is trial 1 with value: 0.8076267281105991.
[I 2026-04-14 15:30:12,837] Trial 3 finished with value: 0.7979641326415521 and parameters: {'n_estimators': 61, 'max_depth': 4, 'min_samples_split': 9}. Best is trial 1 with value: 0.8076267281105991.
[I 2026-04-14 15:30:12,916] Trial 4 finished with value: 0.802147920347226

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
24,24,0.817682,2026-04-14 15:30:14.465636,2026-04-14 15:30:14.543425,0 days 00:00:00.077789,7,9,62,COMPLETE
95,95,0.817682,2026-04-14 15:30:24.168968,2026-04-14 15:30:24.342163,0 days 00:00:00.173195,14,9,169,COMPLETE
77,77,0.817682,2026-04-14 15:30:21.535497,2026-04-14 15:30:21.725060,0 days 00:00:00.189563,16,9,167,COMPLETE
43,43,0.814928,2026-04-14 15:30:15.915952,2026-04-14 15:30:15.963952,0 days 00:00:00.048000,15,10,21,COMPLETE
97,97,0.811660,2026-04-14 15:30:24.517536,2026-04-14 15:30:24.706249,0 days 00:00:00.188713,17,9,164,COMPLETE


In [63]:
# Train using optimal parameters
model_2 = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_f1_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_f1_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_f1_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model_2.fit(X_train, y_train)

,n_estimators,62
,criterion,'gini'
,max_depth,7
,min_samples_split,9
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [64]:
y_pred = model_2.predict(X_test)

In [65]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       190
           1       0.79      0.79      0.79        19

    accuracy                           0.96       209
   macro avg       0.88      0.88      0.88       209
weighted avg       0.96      0.96      0.96       209



In [66]:
f1_model_report_df = classification_report(y_test, y_pred, output_dict=True)

### Analyze reports

In [67]:
# 2. Convert and transpose
recall_report_df = pd.DataFrame(recall_model_report_df).transpose()
f1_report_df = pd.DataFrame(f1_model_report_df).transpose()

# 3. Combine multiple reports (e.g., adding a 'Model' column for identification)
combined_df = pd.concat([recall_report_df, f1_report_df], keys=['RF optimized for Recall', 'RF optimized for F1'])

In [68]:
combined_df

precision    recall  f1-score  \
RF optimized for Recall 0              0.988571  0.910526  0.947945   
                        1              0.500000  0.894737  0.641509   
                        accuracy       0.909091  0.909091  0.909091   
                        macro avg      0.744286  0.902632  0.794727   
                        weighted avg   0.944156  0.909091  0.920087   
RF optimized for F1     0              0.978947  0.978947  0.978947   
                        1              0.789474  0.789474  0.789474   
                        accuracy       0.961722  0.961722  0.961722   
                        macro avg      0.884211  0.884211  0.884211   
                        weighted avg   0.961722  0.961722  0.961722   

                                         support  
RF optimized for Recall 0             190.000000  
                        1              19.000000  
                        accuracy        0.909091  
                        macro avg     209.000000  
                        weighted avg  209.000000  
RF optimized for F1     0             190.000000  
                        1              19.000000  
                        accuracy        0.961722  
                        macro avg     209.000000  
                        weighted avg  209.000000

In [70]:
# Recall model
importances = model_1.feature_importances_
feature_names = df.drop(columns='Fraud').columns
feature_imp_recall_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_recall_df

,Feature,Gini Importance
24,ForgivenessAmount,0.205954
26,NotForgivenAmount,0.152107
39,ForgivenessDate_int,0.121225
27,ForgivenPercentage,0.117197
40,LoanStatusDate_int,0.104449
2,LoanStatus,0.100486
38,DateApproved_int,0.037241
3,Term,0.031180
37,PROCEED_Per_Job,0.031100
0,ProcessingMethod,0.027298


In [71]:
# Recall model
importances = model_2.feature_importances_
feature_names = df.drop(columns='Fraud').columns
feature_imp_f1_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_f1_df

,Feature,Gini Importance
24,ForgivenessAmount,1.763943e-01
26,NotForgivenAmount,1.529056e-01
39,ForgivenessDate_int,1.351834e-01
27,ForgivenPercentage,1.065363e-01
40,LoanStatusDate_int,9.683011e-02
2,LoanStatus,7.074393e-02
38,DateApproved_int,5.664837e-02
3,Term,2.198516e-02
0,ProcessingMethod,1.982580e-02
37,PROCEED_Per_Job,1.874965e-02
